# 1. Importando as Bibliotecas

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing import image_dataset_from_directory
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# 2. Carregamento e Pré-processamento dos Dados

In [ ]:
# Definindo hiperparâmetros
BATCH_SIZE = 32
IMG_SIZE = (224, 224) # Tamanho exigido pela MobileNetV2
EPOCHS = 15

print("Carregando base de Treino...")
train_dataset = image_dataset_from_directory(
    'train',
    shuffle=True,
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE,
    label_mode='binary'
)

print("Carregando base de Validação...")
valid_dataset = image_dataset_from_directory(
    'valid',
    shuffle=True,
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE,
    label_mode='binary'
)

print("Carregando base de Teste...")
test_dataset = image_dataset_from_directory(
    'test',
    shuffle=False, # Não embaralhar para podermos bater com a matriz de confusão depois
    batch_size=BATCH_SIZE,
    image_size=IMG_SIZE,
    label_mode='binary'
)

# Otimização de I/O (carregamento mais rápido)
AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.prefetch(buffer_size=AUTOTUNE)
valid_dataset = valid_dataset.prefetch(buffer_size=AUTOTUNE)
test_dataset = test_dataset.prefetch(buffer_size=AUTOTUNE)

# 3. Construção do Modelo (Transfer Learning)

In [ ]:
# Pré-processamento específico da MobileNetV2 (escala os pixels para entre -1 e 1)
preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input

# Instanciando o modelo base pré-treinado no ImageNet
base_model = MobileNetV2(
    input_shape=IMG_SIZE + (3,),
    include_top=False, # Removemos a camada final original
    weights='imagenet'
)

# Congelamos a base para não destruir os pesos pré-treinados
base_model.trainable = False

# Construindo o nosso topo customizado
inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = preprocess_input(inputs) # Aplica normalização
x = base_model(x, training=False) 
x = GlobalAveragePooling2D()(x) # Achata as características
x = Dropout(0.2)(x) # Previne overfitting

# Camada final de saída com Sigmoid (retorna 0 a 1)
# 0 pode ser 'failure' e 1 pode ser 'success' (depende da ordem das pastas)
outputs = Dense(1, activation='sigmoid')(x)

model = Model(inputs, outputs)

model.summary()

# 4. Compilação e Treinamento

In [ ]:
# Compilando o modelo
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Callbacks para salvar o melhor modelo e parar cedo se não melhorar
callbacks = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    ModelCheckpoint('melhor_modelo_impressao3d.keras', monitor='val_accuracy', save_best_only=True)
]

# Iniciando o treinamento
history = model.fit(
    train_dataset,
    validation_data=valid_dataset,
    epochs=EPOCHS,
    callbacks=callbacks
)

# 5. Avaliação e Extração das Métricas (Para o Relatório/Apresentação)

In [ ]:
# 1. Plotando as Curvas de Aprendizado (Exigência da atividade)
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Treino')
plt.plot(history.history['val_accuracy'], label='Validação')
plt.title('Acurácia (Accuracy)')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Treino')
plt.plot(history.history['val_loss'], label='Validação')
plt.title('Perda (Loss)')
plt.legend()
plt.show()

# 2. Avaliando no conjunto de Teste Invisível
print("Avaliando no conjunto de Teste...")
loss, accuracy = model.evaluate(test_dataset)
print(f'Acurácia no Teste: {accuracy*100:.2f}%')

# 3. Gerando Matriz de Confusão, Precision, Recall e F1-Score
y_true = np.concatenate([y for x, y in test_dataset], axis=0)
y_pred_probs = model.predict(test_dataset)
y_pred = (y_pred_probs > 0.5).astype("int32")

# Nomes das classes geradas automaticamente pelo Keras (Geralmente alfabético)
# Como você tem 'failure' e 'success', class_names = ['failure', 'success']
print("\n--- Relatório de Classificação ---")
print(classification_report(y_true, y_pred, target_names=['Falha (0)', 'Sucesso (1)']))

# Plotando a Matriz de Confusão
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Falha', 'Sucesso'], yticklabels=['Falha', 'Sucesso'])
plt.ylabel('Rótulo Real')
plt.xlabel('Predição do Modelo')
plt.title('Matriz de Confusão')
plt.show()